# Setup

Firefly enables interactive 3D view of any particle set (positions + optional per-particle fields) in the browser, using [Firefly](https://firefly.rcs.northwestern.edu/) ([GitHub](https://github.com/ageller/Firefly)).

## Installation

Use a fresh environment so Firefly's web dependencies (flask, flask-socketio, eventlet) don't clash with your analysis stack:

```bash
conda create -n firefly python=3.11.16 -y
conda activate firefly
pip install firefly scipy tqdm ipykernel
python -m ipykernel install --user --name firefly --display-name "firefly"
```

Then pick the **firefly** kernel for this notebook. Check the install with:

```bash
python -c "from firefly.data_reader import Reader; print('ok')"
```

Notes:
- The PyPI package is `firefly`. The installation pulls in numpy, h5py, pandas, flask, flask-socketio, eventlet, requests, and abg_python.
- Grab `tqdm` if you add progress bars.
- Eventlet prints deprecation warnings on import. They are harmless; the setup cell will silence them.

## Which server cell to run
- Notebook running **on your own laptop**: run the *local* server cell.
- Notebook running **on a remote machine** (cluster, ssh'd Jupyter): run the *remote* server cell and open the SSH tunnel it prints.

## Import

In [ ]:
import os, socket, shutil, getpass, warnings
import numpy as np
from importlib.metadata import version
from firefly.data_reader import Reader, ParticleGroup

# get rid of eventlet deprecation spam
warnings.filterwarnings("ignore")

# where the firefly files are written
# change to any writable directory
OUT_DIR = os.path.expanduser("firefly_out")

# name shown in the firefly data picker
# also name of output dataset
DATASET = "my_dataset"
# for downsampling data
SEED = 42

# make directory of writable
os.makedirs(OUT_DIR, exist_ok=True)
print(f"firefly {version('firefly')} on {socket.gethostname()}")

# Load data

Loading data is simple

Just pass a set of positions and a set of fields to each group

```python
# from numpy
d = np.load("particles.npz")
groups = {"DM": {"pos": d["pos"], "fields": {"vx": d["vel"][:, 0]}}}

# from hdf
import h5py
with h5py.File("snap.hdf5") as f:
    groups = {"DM": {"pos": f["PartType1/Coordinates"][:], "fields": {}}}
```

In [ ]:
"""

---- example data: a dense clump inside a uniform background. replace with yours ----

"""
# data
rng = np.random.default_rng(SEED)
bg = rng.uniform(-50, 50, size=(200_000, 3))
clump = rng.normal(0, 5, size=(50_000, 3))


# data group passed to firefly
# here background & clump are initialized as two separate groups
# each group can have its own "pos" and "fields"
groups = {
    "background": {
        "pos": bg,
        "fields": {"radius": np.linalg.norm(bg, axis=1)},
    },
    "clump": {
        "pos": clump,
        "fields": {"radius": np.linalg.norm(clump, axis=1),
                   "random": rng.random(len(clump))},  # a random field
    },
}


## [Optional] Subsample data

We subsample the dataset to a set number of particles per group

Also write the subsampled dataset to disk, so you can rerun this cheaply, or download and run locally if you'd like

In [ ]:
MAX_PER_GROUP = 1_000_000     # browser budget; ~1E6 particles runs smooth on my potato laptop
CENTER = "auto"               # "auto" = center of all groups' bounding box, None = as-is, or (x, y, z)

# one shared shift, so groups stay aligned with each other
if CENTER == "auto":
    allpos = np.concatenate([np.asarray(g["pos"]) for g in groups.values()])
    shift = 0.5 * (allpos.min(axis=0) + allpos.max(axis=0))
    del allpos
elif CENTER is None:
    shift = np.zeros(3)
else:
    shift = np.asarray(CENTER, dtype=float)

datadir = os.path.join(OUT_DIR, DATASET)
shutil.rmtree(datadir, ignore_errors=True)
reader = Reader(datadir=datadir, write_startup=True)


# randomly subsample your data
rng = np.random.default_rng(SEED)
for name, g in groups.items():
    pos = np.asarray(g["pos"], dtype=np.float64)
    fields = {k: np.asarray(v) for k, v in g.get("fields", {}).items()}

    # sanity checks
    assert pos.ndim == 2 and pos.shape[1] == 3, f"{name}: pos must be (N, 3), got {pos.shape}"
    for k, v in fields.items():
        assert v.shape == (len(pos),), f"{name}/{k}: length {v.shape} != {len(pos)}"

    # drop rows with non-finite position or field values
    ok = np.isfinite(pos).all(axis=1)
    for v in fields.values():
        ok &= np.isfinite(v)

    idx = np.flatnonzero(ok)
    if len(idx) > MAX_PER_GROUP:
        idx = np.sort(rng.choice(idx, size=MAX_PER_GROUP, replace=False))

    nf = len(fields)
    reader.addParticleGroup(ParticleGroup(
        name,
        (pos[idx] - shift).astype(np.float32),
        field_arrays=[v[idx].astype(np.float32) for v in fields.values()],
        field_names=list(fields),
        field_filter_flags=[True] * nf,
        field_colormap_flags=[True] * nf,
        field_radius_flags=[False] * nf,
    ))
    print(f"{name}: {len(idx):,} of {len(pos):,} kept "
          f"({len(pos) - ok.sum():,} non-finite dropped), fields {list(fields)}")

reader.writeToDisk(loud=False)
print(f"\nwrote to {datadir}, shift applied = {shift}")

# Start server [LOCALLY]

If you are running this notebook locally, the Firefly server setup is easy

In [ ]:
import webbrowser, requests
from firefly.server import spawnFireflyServer, quitAllFireflyServers

quitAllFireflyServers()              # clear servers left from a previous run
PORT = free_port()                   # helper defined in the remote cell; or set PORT = 5500
process = spawnFireflyServer(port=PORT)

r = requests.get(f"http://localhost:{PORT}/", timeout=10)
url = f"http://localhost:{PORT}"
print(f"server -> HTTP {r.status_code}, open {url}")
webbrowser.open(url)                 # opens a tab; harmless if it can't

# Start server [ON CLUSTER]

In [ ]:
import requests
from firefly.server import spawnFireflyServer, quitAllFireflyServers

def free_port(start=5500, stop=5600):
    """first open port in range; on a shared login node someone else may hold 5500"""
    for p in range(start, stop):
        with socket.socket() as s:
            try:
                s.bind(("", p))
                return p
            except OSError:
                continue
    raise RuntimeError("no free port found")

quitAllFireflyServers()
PORT = free_port()
process = spawnFireflyServer(port=PORT)

r = requests.get(f"http://localhost:{PORT}/", timeout=10)
node = socket.gethostname().split(".")[0]
print(f"server on {node}:{PORT} -> HTTP {r.status_code}")
print(f"""

notebook on your own machine:  open http://localhost:{PORT}

notebook on a remote machine, run this on your LAPTOP, then open http://localhost:{PORT}
    ssh -N -L {PORT}:{node}:{PORT} {getpass.getuser()}@<login-host>
""")

In [ ]:
quitAllFireflyServers()

In [ ]:
bundle = os.path.join(OUT_DIR, f"{DATASET}_bundle")
shutil.rmtree(bundle, ignore_errors=True)
reader.copyFireflySourceToTarget(target=bundle)
print(f"""
copy {bundle} to your laptop, then
    cd {os.path.basename(bundle)} && python3 -m http.server 8000
and open http://localhost:8000
""")